# 05. Unsupervised Player Clustering & Archetypes

**Football IQ - Player Performance Prediction & Scouting System**

K-Means clustering, Elbow method, Silhouette analysis, and 2D/3D PCA dimensionality reduction to discover tactical player archetypes.


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

import sys
sys.path.append("..")
from src.model_training import train_player_clustering, CLUSTERING_FEATURES, ARCHETYPE_NAMES

# Load processed data
df = pd.read_csv(os.path.join("..", "data", "processed", "players_processed.csv"))
kmeans, c_scaler, pca_2d, meta = train_player_clustering(df, n_clusters=5)

print(f"Silhouette Score: {meta['silhouette_score']}")
print("Cluster Profiles:")
for c_id, prof in meta['cluster_profiles'].items():
    print(f" - Cluster {c_id} ({prof['name']}): {prof['count']} players")


### 1. Elbow Method & Inertia Plot


In [ ]:
feat_cols = [c for c in CLUSTERING_FEATURES if c in df.columns]
X_scaled = c_scaler.transform(df[feat_cols].fillna(0))

inertias = []
k_range = range(2, 10)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertias, "bo-")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia (Sum of Squared Distances)")
plt.title("Elbow Method for Optimal k")
plt.grid(True)
plt.show()


### 2. 2D PCA Cluster Scatter Plot


In [ ]:
pca_coords = pca_2d.transform(X_scaled)
df_plot = df.copy()
df_plot["cluster_id"] = kmeans.predict(X_scaled)
df_plot["archetype"] = df_plot["cluster_id"].map(ARCHETYPE_NAMES)
df_plot["PCA_1"] = pca_coords[:, 0]
df_plot["PCA_2"] = pca_coords[:, 1]

plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_plot, x="PCA_1", y="PCA_2", hue="archetype", palette="Set1", alpha=0.7)
plt.title("2D PCA Projection of Tactical Player Clusters")
plt.grid(True, alpha=0.3)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()
